# File 06 LR — Lighting-Robust Test Evaluation
Locked test set. No threshold tuning on test.


In [1]:
import os, json, random
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, confusion_matrix, brier_score_loss
from sklearn.calibration import calibration_curve
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models import efficientnet_b0

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MANIFEST_PATH=Path(r'D:\DIABETES\diabetes_pipeline_outputs\03_segmented_export_manifest.csv')
CHECKPOINT=Path(r'D:\DIABETES\diabetes_pipeline_outputs\05_segmented_training_lighting_robust\best_segmented_lighting_robust_model.pth')
SUMMARY_JSON=Path(r'D:\DIABETES\diabetes_pipeline_outputs\05_segmented_training_lighting_robust\05_lr_best_checkpoint_summary.json')
OUTPUT_DIR=Path(r'D:\DIABETES\diabetes_pipeline_outputs\06_segmented_test_evaluation_lighting_robust')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMG_SIZE=224; BATCH_SIZE=32; N_BOOT=1000
MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
print(f'Device: {DEVICE}')


Device: cuda


## Lighting Correction + Load Data


In [2]:
def correct_lighting_clahe(image_pil, clip_limit=1.5, tile_grid_size=(8,8)):
    img_rgb=np.array(image_pil.convert('RGB'))
    lab=cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l,a,b=cv2.split(lab)
    clahe=cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    l_eq=clahe.apply(l)
    lab_eq=cv2.merge([l_eq,a,b])
    return Image.fromarray(cv2.cvtColor(lab_eq, cv2.COLOR_LAB2RGB))

def pad_square(img):
    w,h=img.size; s=max(w,h); new=Image.new('RGB',(s,s),(0,0,0)); new.paste(img,((s-w)//2,(s-h)//2)); return new

df=pd.read_csv(MANIFEST_PATH)
df=df[df['export_status']=='ok'].reset_index(drop=True)
img_col='segmented_image_path' if 'segmented_image_path' in df.columns else 'crop_masked_path'
df['image_path']=df[img_col]
df_test=df[df['final_split']=='test'].reset_index(drop=True)

if SUMMARY_JSON.exists():
    with open(SUMMARY_JSON) as f: bsum=json.load(f)
    SELECTED_THRESHOLD=float(bsum.get('best_threshold',0.5))
    print(f'Loaded threshold: {SELECTED_THRESHOLD:.3f}')
else:
    SELECTED_THRESHOLD=0.5; print('WARNING: fallback threshold 0.5')

print(f'Test images: {len(df_test)}')


Loaded threshold: 0.500
Test images: 412


## Model and Dataset


In [3]:
transform=T.Compose([T.Lambda(correct_lighting_clahe),T.Lambda(pad_square),T.Resize((IMG_SIZE,IMG_SIZE)),T.ToTensor(),T.Normalize(mean=MEAN,std=STD)])

class TestDS(Dataset):
    def __init__(self,df,t): self.df=df.reset_index(drop=True); self.t=t
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        row=self.df.iloc[i]; img=Image.open(row['image_path']).convert('RGB')
        return self.t(img),int(row['label_binary']),i

model=efficientnet_b0(weights=None)
model.classifier=nn.Sequential(nn.Dropout(0.3),nn.Linear(model.classifier[1].in_features,1))
model.load_state_dict(torch.load(CHECKPOINT,map_location=DEVICE))
model=model.to(DEVICE); model.eval(); print('Model loaded.')
ds=TestDS(df_test,transform); dl=DataLoader(ds,BATCH_SIZE,shuffle=False,num_workers=0)


Model loaded.


## Inference and Metrics


In [4]:
yt,yp,idx_list=[],[],[]
with torch.no_grad():
    for imgs,labels,idx in dl:
        out=model(imgs.to(DEVICE)); prob=torch.sigmoid(out).cpu().numpy().flatten()
        yt.extend(labels.numpy()); yp.extend(prob); idx_list.extend(idx.numpy())
yt=np.array(yt); yp=np.array(yp)
df_pred=df_test.iloc[idx_list].copy()
df_pred['y_true']=yt; df_pred['y_prob_diabetes']=yp
df_pred['y_pred_0_5']=(yp>=0.5).astype(int); df_pred['y_pred_sel']=(yp>=SELECTED_THRESHOLD).astype(int)
df_pred.to_csv(OUTPUT_DIR/'06_lr_test_predictions.csv',index=False)
print('Predictions saved.')


Predictions saved.


In [5]:
def compute_metrics(yt,yp,thr=0.5):
    pred=(yp>=thr).astype(int); tn,fp,fn,tp=confusion_matrix(yt,pred,labels=[0,1]).ravel()
    recall=tp/(tp+fn) if (tp+fn)>0 else 0; spec=tn/(tn+fp) if (tn+fp)>0 else 0
    ppv=tp/(tp+fp) if (tp+fp)>0 else 0; npv=tn/(tn+fn) if (tn+fn)>0 else 0
    f1=2*ppv*recall/(ppv+recall) if (ppv+recall)>0 else 0
    return {'tp':int(tp),'tn':int(tn),'fp':int(fp),'fn':int(fn),'diabetes_recall':recall,'fnr':fn/(tp+fn) if (tp+fn)>0 else 0,'specificity':spec,'ppv':ppv,'npv':npv,'f1':f1,'balanced_accuracy':(recall+spec)/2,'accuracy':(tp+tn)/(tp+tn+fp+fn),'brier':brier_score_loss(yt,yp)}

m05=compute_metrics(yt,yp,0.5); msel=compute_metrics(yt,yp,SELECTED_THRESHOLD)
pd.DataFrame([m05]).to_csv(OUTPUT_DIR/'06_lr_test_metrics_threshold_0_5.csv',index=False)
pd.DataFrame([msel]).to_csv(OUTPUT_DIR/'06_lr_test_metrics_selected_threshold.csv',index=False)
roc_auc=roc_auc_score(yt,yp); pr_auc=average_precision_score(yt,yp)
pd.DataFrame([{'roc_auc':roc_auc,'pr_auc':pr_auc,'prob_mean_1':yp[yt==1].mean(),'prob_mean_0':yp[yt==0].mean()}]).to_csv(OUTPUT_DIR/'06_lr_test_probability_metrics.csv',index=False)
print(f'ROC:{roc_auc:.4f} PR:{pr_auc:.4f}')


ROC:0.9601 PR:0.9624


In [6]:
# Bootstrap CIs
rng=np.random.default_rng(SEED); n=len(yt)
ci_rows=[]
for name,fn in [('recall',lambda a,b:compute_metrics(a,b,SELECTED_THRESHOLD)['diabetes_recall']),('fnr',lambda a,b:compute_metrics(a,b,SELECTED_THRESHOLD)['fnr']),('specificity',lambda a,b:compute_metrics(a,b,SELECTED_THRESHOLD)['specificity']),('ppv',lambda a,b:compute_metrics(a,b,SELECTED_THRESHOLD)['ppv']),('npv',lambda a,b:compute_metrics(a,b,SELECTED_THRESHOLD)['npv']),('f1',lambda a,b:compute_metrics(a,b,SELECTED_THRESHOLD)['f1']),('balanced_accuracy',lambda a,b:compute_metrics(a,b,SELECTED_THRESHOLD)['balanced_accuracy']),('accuracy',lambda a,b:compute_metrics(a,b,SELECTED_THRESHOLD)['accuracy']),('roc_auc',lambda a,b:roc_auc_score(a,b)),('pr_auc',lambda a,b:average_precision_score(a,b)),('brier',lambda a,b:brier_score_loss(a,b))]:
    vals=[fn(yt[rng.choice(n,n,replace=True)],yp[rng.choice(n,n,replace=True)]) for _ in range(N_BOOT)]
    lo,hi=np.percentile(vals,[2.5,97.5])
    ci_rows.append({'metric':name,'ci_2.5':lo,'ci_97.5':hi})
pd.DataFrame(ci_rows).to_csv(OUTPUT_DIR/'06_lr_test_bootstrap_confidence_intervals.csv',index=False)
print('Bootstrap CIs saved.')


Bootstrap CIs saved.


In [7]:
# Subgroup
subgroups=[{'group':'all_test','count':len(yt),'unstable':False,**msel}]
for col,flag in [('suspicious_edge_touch',True),('suspicious_edge_touch',False),('suspicious_large_mask',True),('segmentation_status','ok')]:
    if col in df_pred.columns:
        mask_sg=df_pred[col]==flag; cnt=mask_sg.sum()
        if cnt>=10:
            m=compute_metrics(yt[mask_sg],yp[mask_sg],SELECTED_THRESHOLD)
            subgroups.append({'group':f'{col}={flag}','count':int(cnt),'unstable':cnt<30,**m})
pd.DataFrame(subgroups).to_csv(OUTPUT_DIR/'06_lr_test_subgroup_metrics.csv',index=False)


In [8]:
# Plots
def plot_cm(yt,yp,thr,title,path):
    cm=confusion_matrix(yt,(yp>=thr).astype(int),labels=[0,1])
    fig,ax=plt.subplots(figsize=(6,6)); ax.matshow(cm,cmap='Blues')
    for i in range(2):
        for j in range(2): ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=14)
    ax.set_title(title); ax.set_xticks([0,1]); ax.set_yticks([0,1]); ax.set_xticklabels(['ND','D']); ax.set_yticklabels(['ND','D'])
    plt.tight_layout(); plt.savefig(path,dpi=100); plt.close()
plot_cm(yt,yp,0.5,'Threshold 0.5',OUTPUT_DIR/'06_lr_confusion_matrix_threshold_0_5.png')
plot_cm(yt,yp,SELECTED_THRESHOLD,f'Threshold {SELECTED_THRESHOLD:.3f}',OUTPUT_DIR/'06_lr_confusion_matrix_selected_threshold.png')
fpr,tpr,_=roc_curve(yt,yp); fig,ax=plt.subplots(figsize=(7,7)); ax.plot(fpr,tpr,label=f'AUC={roc_auc:.3f}'); ax.plot([0,1],[0,1],'k--'); ax.set_title('ROC'); ax.legend(); plt.tight_layout(); plt.savefig(OUTPUT_DIR/'06_lr_roc_curve.png',dpi=100); plt.close()
prec,rec,_=precision_recall_curve(yt,yp); fig,ax=plt.subplots(figsize=(7,7)); ax.plot(rec,prec,label=f'PR-AUC={pr_auc:.3f}'); ax.set_title('PR Curve'); ax.legend(); plt.tight_layout(); plt.savefig(OUTPUT_DIR/'06_lr_pr_curve.png',dpi=100); plt.close()
fp2,mp2=calibration_curve(yt,yp,n_bins=10); fig,ax=plt.subplots(figsize=(7,7)); ax.plot(mp2,fp2,'s-',label='Model'); ax.plot([0,1],[0,1],'k--'); ax.set_title('Calibration'); ax.legend(); plt.tight_layout(); plt.savefig(OUTPUT_DIR/'06_lr_calibration_curve.png',dpi=100); plt.close()
fig,ax=plt.subplots(figsize=(8,5)); ax.hist(yp[yt==0],bins=20,alpha=0.5,label='ND'); ax.hist(yp[yt==1],bins=20,alpha=0.5,label='D'); ax.axvline(SELECTED_THRESHOLD,color='r',linestyle='--',label=f't={SELECTED_THRESHOLD:.3f}'); ax.legend(); plt.tight_layout(); plt.savefig(OUTPUT_DIR/'06_lr_probability_histogram_by_class.png',dpi=100); plt.close()
print('Plots saved.')


Plots saved.


In [9]:
# Compare to old File 06
old_m_path=Path(r'D:\DIABETES\diabetes_pipeline_outputs\06_segmented_test_evaluation\06_segmented_test_metrics_selected_threshold.csv')
comp_str=''
if old_m_path.exists():
    old_m=pd.read_csv(old_m_path)
    comp_str=f'\nComparison to previous File 06:\nOld recall:{old_m["diabetes_recall"].iloc[0]:.4f} New:{msel["diabetes_recall"]:.4f}\nOld ROC:{pd.read_csv(str(old_m_path).replace("selected_threshold","probability_metrics"))["roc_auc"].iloc[0] if Path(str(old_m_path).replace("selected_threshold","probability_metrics")).exists() else "N/A"} New:{roc_auc:.4f}'
json.dump({'checkpoint':str(CHECKPOINT),'threshold':SELECTED_THRESHOLD,'test_count':len(yt),'roc_auc':roc_auc,'pr_auc':pr_auc,'metrics_05':m05,'metrics_sel':msel},open(OUTPUT_DIR/'06_lr_test_config.json','w'),indent=2,default=str)
pd.DataFrame([{'split':'test','count':len(df_test),'diabetes':int((df_test['label_binary']==1).sum()),'non_diabetes':int((df_test['label_binary']==0).sum())}]).to_csv(OUTPUT_DIR/'06_lr_test_dataset_summary.csv',index=False)
handoff=f'FILE 06 LR HANDOFF\nStatus: PASS\nCheckpoint:{CHECKPOINT}\nThreshold:{SELECTED_THRESHOLD:.3f} (from validation)\nTest:{len(yt)} D:{(yt==1).sum()} ND:{(yt==0).sum()}\nRecall:{msel["diabetes_recall"]:.4f} Spec:{msel["specificity"]:.4f} ROC:{roc_auc:.4f} PR:{pr_auc:.4f}\nNo threshold tuning on test.{comp_str}\nRecommendation: Proceed to File 07\n'
with open(OUTPUT_DIR/'06_lr_test_handoff_summary.txt','w') as f: f.write(handoff)
print(handoff)


FILE 06 LR HANDOFF
Status: PASS
Checkpoint:D:\DIABETES\diabetes_pipeline_outputs\05_segmented_training_lighting_robust\best_segmented_lighting_robust_model.pth
Threshold:0.500 (from validation)
Test:412 D:215 ND:197
Recall:0.9209 Spec:0.8832 ROC:0.9601 PR:0.9624
No threshold tuning on test.
Recommendation: Proceed to File 07

